In [1]:
import os
import requests
import pandas as pd
from datetime import date, datetime
import pytz

# === Get today's date in Eastern time for consistency ===
eastern = pytz.timezone('US/Eastern')
today = datetime.now(eastern).date()
today_str = today.isoformat()

def fetch_starting_pitchers(target_date):
    """
    Fetches the probable starting pitchers for each MLB game on the given date.
    """
    date_str = target_date.isoformat()
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}&hydrate=probablePitcher(note,stats,person)'

    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Failed to fetch data: {response.status_code}")
        return None

    data = response.json()
    games = data.get('dates', [])[0].get('games', []) if data.get('dates') else []

    pitcher_data = []
    for game in games:
        home_team = game['teams']['home']['team']['name']
        away_team = game['teams']['away']['team']['name']

        home_pitcher = game['teams']['home'].get('probablePitcher', {}).get('fullName', 'TBD')
        away_pitcher = game['teams']['away'].get('probablePitcher', {}).get('fullName', 'TBD')

        pitcher_data.append({
            'Date': date_str,
            'Away Team': away_team,
            'Away Starter': away_pitcher,
            'Home Team': home_team,
            'Home Starter': home_pitcher
        })

    return pd.DataFrame(pitcher_data)

# === MAIN ===
df = fetch_starting_pitchers(today)

if df is not None and not df.empty:
    output_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "starting-pitchers"))
    os.makedirs(output_dir, exist_ok=True)

    output_file = os.path.join(output_dir, f"starting_pitchers_{today_str}.csv")
    df.to_csv(output_file, index=False)

    print(f"✅ Starting pitchers saved to {output_file}")
    print(df.head())
else:
    print("⚠️ No starting pitcher data found.")


✅ Starting pitchers saved to /workspaces/MLB_Model/data/starting-pitchers/starting_pitchers_2025-05-29.csv
         Date             Away Team      Away Starter              Home Team  \
0  2025-05-29        Atlanta Braves  AJ Smith-Shawver  Philadelphia Phillies   
1  2025-05-29        Atlanta Braves        Chris Sale  Philadelphia Phillies   
2  2025-05-29             Athletics       Jacob Lopez      Toronto Blue Jays   
3  2025-05-29        Tampa Bay Rays         Shane Baz         Houston Astros   
4  2025-05-29  Washington Nationals    MacKenzie Gore       Seattle Mariners   

         Home Starter  
0  Cristopher Sánchez  
1        Zack Wheeler  
2        José Berríos  
3          Ryan Gusto  
4     Emerson Hancock  
